[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/SFM/blob/main/Quantlets/Ch_02/SFM_ch2_extreme_value/SFM_ch2_extreme_value.ipynb)

# SFM_ch2_extreme_value

Extreme Value Theory for Financial Risk
Description:
- Block maxima approach with GEV distribution
- Peaks Over Threshold (POT) with GPD
- Mean excess function and threshold selection
- Return levels and return periods
- EVT-based VaR and Expected Shortfall
Statistics of Financial Markets course

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
from scipy import stats
from scipy.stats import genextreme, genpareto
import os
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Chart style settings - Nature journal quality
plt.rcParams['figure.facecolor'] = 'none'
plt.rcParams['axes.facecolor'] = 'none'
plt.rcParams['savefig.facecolor'] = 'none'
plt.rcParams['savefig.transparent'] = True
plt.rcParams['axes.grid'] = False
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Helvetica', 'Arial', 'DejaVu Sans']
plt.rcParams['font.size'] = 8
plt.rcParams['axes.labelsize'] = 9
plt.rcParams['axes.titlesize'] = 10
plt.rcParams['xtick.labelsize'] = 8
plt.rcParams['ytick.labelsize'] = 8
plt.rcParams['legend.fontsize'] = 8
plt.rcParams['legend.facecolor'] = 'none'
plt.rcParams['legend.framealpha'] = 0
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.linewidth'] = 0.5
plt.rcParams['lines.linewidth'] = 0.75

# Color palette
MAIN_BLUE = '#1A3A6E'
CRIMSON = '#DC3545'
FOREST = '#2E7D32'
AMBER = '#B5853F'
ORANGE = '#E67E22'

# Output directory
CHART_DIR = os.path.join('..', '..', '..', 'charts')
os.makedirs(CHART_DIR, exist_ok=True)

def save_fig(name):
    """Save figure with transparent background."""
    plt.savefig(os.path.join(CHART_DIR, f'{name}.pdf'),
                bbox_inches='tight', transparent=True)
    plt.savefig(os.path.join(CHART_DIR, f'{name}.png'),
                bbox_inches='tight', transparent=True, dpi=300)
    plt.close()
    print(f"   Saved: {name}.pdf/.png")

## Download Data

In [ ]:
# Download S&P 500 data
data = yf.download('^GSPC', start='2000-01-01', end='2025-01-01', progress=False)
close = data['Close'].squeeze()
log_ret = np.log(close / close.shift(1)).dropna()

# Losses = negative log-returns (positive values represent losses)
losses = -log_ret

print(f"   Period: {close.index[0].strftime('%Y-%m-%d')} to {close.index[-1].strftime('%Y-%m-%d')}")
print(f"   Observations: {len(log_ret)}")
print(f"   Mean log-return: {log_ret.mean():.6f}")
print(f"   Std log-return:  {log_ret.std():.6f}")
print()
print(f"   Loss statistics (losses = -log_returns):")
print(f"     Mean:     {losses.mean():.6f}")
print(f"     Std:      {losses.std():.6f}")
print(f"     Max loss: {losses.max():.6f}")
print(f"     Min loss: {losses.min():.6f}")
print(f"     Skewness: {stats.skew(losses):.4f}")
print(f"     Kurtosis: {stats.kurtosis(losses):.4f}")

## Block Maxima and GEV Fit

In [ ]:
# Group losses into monthly blocks and take the maximum of each block
losses_df = pd.DataFrame({'loss': losses})
block_maxima = losses_df.resample('ME').max().dropna().squeeze()

print(f"   Number of monthly blocks: {len(block_maxima)}")
print(f"   Block maxima - Mean: {block_maxima.mean():.6f}, Std: {block_maxima.std():.6f}")
print()

# Fit GEV distribution to block maxima
# scipy.stats.genextreme uses sign convention: c = -xi
gev_params = genextreme.fit(block_maxima)
c_gev, loc_gev, scale_gev = gev_params
xi_gev = -c_gev  # shape parameter in standard EVT notation

print(f"   GEV fit (standard EVT notation):")
print(f"     xi (shape):    {xi_gev:.4f}")
print(f"     mu (location): {loc_gev:.6f}")
print(f"     sigma (scale): {scale_gev:.6f}")
print()

# Plot histogram of block maxima with GEV and Normal overlay
fig, ax = plt.subplots(figsize=(7, 3.5))

ax.hist(block_maxima, bins=40, density=True, alpha=0.5, color=MAIN_BLUE,
        edgecolor='white', label='Block maxima')

x_gev = np.linspace(block_maxima.min() * 0.8, block_maxima.max() * 1.1, 500)
ax.plot(x_gev, genextreme.pdf(x_gev, *gev_params), color=CRIMSON,
        linewidth=1.5, label=f'GEV ($\\xi$={xi_gev:.3f})')

# Normal comparison
norm_loc, norm_scale = stats.norm.fit(block_maxima)
ax.plot(x_gev, stats.norm.pdf(x_gev, norm_loc, norm_scale), color=FOREST,
        linewidth=1.2, linestyle='--', label='Normal')

ax.set_xlabel('Monthly block maximum loss')
ax.set_ylabel('Density')
ax.set_title('Block Maxima with GEV Fit', fontweight='bold')
ax.legend(loc='upper right', fontsize=7, frameon=False)

# Annotate fitted parameters
ax.text(0.98, 0.65,
        f'$\\xi$ = {xi_gev:.4f}\n$\\mu$ = {loc_gev:.4f}\n$\\sigma$ = {scale_gev:.4f}',
        transform=ax.transAxes, fontsize=7, va='top', ha='right',
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

plt.tight_layout()
save_fig('ch2_evt_gev_fit')

## Peaks Over Threshold (POT)

In [ ]:
# Set threshold at the 95th percentile of losses
threshold = np.percentile(losses, 95)
print(f"   Threshold (95th percentile): {threshold:.6f}")

# Extract exceedances (losses above threshold)
excess_idx = losses > threshold
exceedances = losses[excess_idx] - threshold  # excess over threshold

print(f"   Number of exceedances: {len(exceedances)}")
print(f"   Exceedance ratio: {len(exceedances)/len(losses):.4f}")
print()

# Fit GPD to exceedances (floc=0 since exceedances are already shifted)
gpd_params = genpareto.fit(exceedances, floc=0)
c_gpd, loc_gpd, scale_gpd = gpd_params
xi_gpd = c_gpd  # GPD shape parameter

print(f"   GPD fit:")
print(f"     xi (shape): {xi_gpd:.4f}")
print(f"     sigma (scale): {scale_gpd:.6f}")
print()

# Plot histogram of exceedances with GPD overlay
fig, ax = plt.subplots(figsize=(7, 3.5))

ax.hist(exceedances, bins=50, density=True, alpha=0.5, color=MAIN_BLUE,
        edgecolor='white', label='Exceedances')

x_gpd = np.linspace(0, exceedances.max() * 1.1, 500)
ax.plot(x_gpd, genpareto.pdf(x_gpd, *gpd_params), color=CRIMSON,
        linewidth=1.5, label=f'GPD ($\\xi$={xi_gpd:.3f})')

ax.set_xlabel('Excess over threshold')
ax.set_ylabel('Density')
ax.set_title(f'Peaks Over Threshold (u = {threshold:.4f})', fontweight='bold')
ax.legend(loc='upper right', fontsize=7, frameon=False)

ax.text(0.98, 0.70,
        f'$\\xi$ = {xi_gpd:.4f}\n$\\sigma$ = {scale_gpd:.4f}\nn = {len(exceedances)}',
        transform=ax.transAxes, fontsize=7, va='top', ha='right',
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

plt.tight_layout()
save_fig('ch2_evt_gpd_pot')

## Mean Excess Function

In [ ]:
# Compute empirical mean excess function: e(u) = E[X - u | X > u]
percentiles = np.linspace(80, 99.5, 200)
thresholds_mef = np.percentile(losses, percentiles)

mean_excess = []
ci_lower = []
ci_upper = []

for u in thresholds_mef:
    exc = losses[losses > u] - u
    n_exc = len(exc)
    if n_exc > 2:
        me = exc.mean()
        se = exc.std() / np.sqrt(n_exc)
        mean_excess.append(me)
        ci_lower.append(me - 1.96 * se)
        ci_upper.append(me + 1.96 * se)
    else:
        mean_excess.append(np.nan)
        ci_lower.append(np.nan)
        ci_upper.append(np.nan)

mean_excess = np.array(mean_excess)
ci_lower = np.array(ci_lower)
ci_upper = np.array(ci_upper)

# Theoretical GPD mean excess: e(u) = (sigma + xi*u) / (1 - xi) for u > threshold
# Using the GPD fitted above at the 95th percentile
gpd_me_theoretical = (scale_gpd + xi_gpd * (thresholds_mef - threshold)) / (1 - xi_gpd)
# Only valid for thresholds above the POT threshold
gpd_me_theoretical[thresholds_mef < threshold] = np.nan

# Plot
fig, ax = plt.subplots(figsize=(7, 3.5))

ax.plot(thresholds_mef, mean_excess, color=MAIN_BLUE, linewidth=1.2,
        label='Empirical mean excess')
ax.fill_between(thresholds_mef, ci_lower, ci_upper, alpha=0.2,
                color=MAIN_BLUE, label='95% confidence band')
ax.plot(thresholds_mef, gpd_me_theoretical, color=CRIMSON, linewidth=1.2,
        linestyle='--', label='GPD theoretical')

# Vertical line at chosen threshold
ax.axvline(x=threshold, color=FOREST, linewidth=1.0, linestyle=':',
           label=f'Threshold (95th pctile = {threshold:.4f})')

ax.set_xlabel('Threshold $u$')
ax.set_ylabel('Mean excess $e(u)$')
ax.set_title('Mean Excess Function', fontweight='bold')
ax.legend(loc='upper left', fontsize=7, frameon=False)

plt.tight_layout()
save_fig('ch2_evt_mean_excess')

## EVT-based VaR and ES

In [ ]:
# Compute VaR and ES at multiple confidence levels
alphas = [0.95, 0.99, 0.999]
n_obs = len(losses)
n_exceed = len(exceedances)
Fu = n_exceed / n_obs  # exceedance probability above threshold

results = {'alpha': [], 'VaR_EVT': [], 'ES_EVT': [],
           'VaR_Normal': [], 'ES_Normal': [],
           'VaR_Empirical': [], 'ES_Empirical': []}

mu_loss = losses.mean()
sigma_loss = losses.std()

for alpha in alphas:
    # --- EVT/GPD (POT) VaR and ES ---
    # VaR_alpha = u + (sigma/xi) * [((n/N_u)*(1-alpha))^(-xi) - 1]
    p = 1 - alpha  # tail probability
    VaR_evt = threshold + (scale_gpd / xi_gpd) * ((Fu / p) ** xi_gpd - 1)
    # ES_alpha = VaR_alpha / (1 - xi) + (sigma - xi * u) / (1 - xi)
    ES_evt = VaR_evt / (1 - xi_gpd) + (scale_gpd - xi_gpd * threshold) / (1 - xi_gpd)

    # --- Normal VaR and ES ---
    VaR_norm = mu_loss + sigma_loss * stats.norm.ppf(alpha)
    # ES for Normal: mu + sigma * phi(z_alpha) / (1 - alpha)
    z_alpha = stats.norm.ppf(alpha)
    ES_norm = mu_loss + sigma_loss * stats.norm.pdf(z_alpha) / (1 - alpha)

    # --- Empirical VaR and ES ---
    VaR_emp = np.percentile(losses, alpha * 100)
    ES_emp = losses[losses >= VaR_emp].mean()

    results['alpha'].append(alpha)
    results['VaR_EVT'].append(VaR_evt)
    results['ES_EVT'].append(ES_evt)
    results['VaR_Normal'].append(VaR_norm)
    results['ES_Normal'].append(ES_norm)
    results['VaR_Empirical'].append(VaR_emp)
    results['ES_Empirical'].append(ES_emp)

# Print results table
print(f"   {'':>8} {'VaR':>30} {'ES':>30}")
print(f"   {'Alpha':>8} {'EVT':>10} {'Normal':>10} {'Empirical':>10} {'EVT':>10} {'Normal':>10} {'Empirical':>10}")
print(f"   {'-'*88}")
for i, alpha in enumerate(alphas):
    print(f"   {alpha:>8.1%} {results['VaR_EVT'][i]:>10.4f} {results['VaR_Normal'][i]:>10.4f} "
          f"{results['VaR_Empirical'][i]:>10.4f} {results['ES_EVT'][i]:>10.4f} "
          f"{results['ES_Normal'][i]:>10.4f} {results['ES_Empirical'][i]:>10.4f}")

# EUR values for 1M portfolio
portfolio = 1_000_000
print(f"\n   EUR values for {portfolio/1e6:.0f}M EUR portfolio:")
print(f"   {'Alpha':>8} {'VaR_EVT':>12} {'VaR_Normal':>12} {'VaR_Emp':>12} "
      f"{'ES_EVT':>12} {'ES_Normal':>12} {'ES_Emp':>12}")
print(f"   {'-'*80}")
for i, alpha in enumerate(alphas):
    print(f"   {alpha:>8.1%} {results['VaR_EVT'][i]*portfolio:>12,.0f} "
          f"{results['VaR_Normal'][i]*portfolio:>12,.0f} "
          f"{results['VaR_Empirical'][i]*portfolio:>12,.0f} "
          f"{results['ES_EVT'][i]*portfolio:>12,.0f} "
          f"{results['ES_Normal'][i]*portfolio:>12,.0f} "
          f"{results['ES_Empirical'][i]*portfolio:>12,.0f}")

# Grouped bar chart comparing methods
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

x_pos = np.arange(len(alphas))
width = 0.25

# VaR comparison
ax = axes[0]
ax.bar(x_pos - width, results['VaR_EVT'], width, color=CRIMSON, alpha=0.85, label='EVT/GPD')
ax.bar(x_pos, results['VaR_Normal'], width, color=MAIN_BLUE, alpha=0.85, label='Normal')
ax.bar(x_pos + width, results['VaR_Empirical'], width, color=FOREST, alpha=0.85, label='Empirical')
ax.set_xlabel('Confidence level')
ax.set_ylabel('VaR (loss fraction)')
ax.set_title('Value at Risk', fontweight='bold')
ax.set_xticks(x_pos)
ax.set_xticklabels([f'{a:.1%}' for a in alphas])
ax.legend(loc='upper left', fontsize=7, frameon=False)

# ES comparison
ax = axes[1]
ax.bar(x_pos - width, results['ES_EVT'], width, color=CRIMSON, alpha=0.85, label='EVT/GPD')
ax.bar(x_pos, results['ES_Normal'], width, color=MAIN_BLUE, alpha=0.85, label='Normal')
ax.bar(x_pos + width, results['ES_Empirical'], width, color=FOREST, alpha=0.85, label='Empirical')
ax.set_xlabel('Confidence level')
ax.set_ylabel('ES (loss fraction)')
ax.set_title('Expected Shortfall', fontweight='bold')
ax.set_xticks(x_pos)
ax.set_xticklabels([f'{a:.1%}' for a in alphas])
ax.legend(loc='upper left', fontsize=7, frameon=False)

plt.tight_layout()
save_fig('ch2_evt_var_es')